# Préparation du catalogue final
Entrée : `film_propre.csv` + `titres_fr.csv` → Sortie : `film_complet.csv`

## 1. Lecture et vérifications

Lecture du fichier de films filtrés, puis contrôle de sa taille et des doublons.

In [15]:
import pandas as pd
import numpy as np

data = pd.read_csv("../data/propre/film_propre.csv")

print("Films :", len(data), "| Colonnes :", data.shape[1], "| Doublons :", data["tconst"].duplicated().sum())

Films : 3508 | Colonnes : 35 | Doublons : 0


## 2. Une seule ligne par film
En cas de fiches TMDB en double, on garde la plus populaire.

In [16]:
data = data.sort_values(["tconst", "popularity"], ascending=[True, False])
data = data.drop_duplicates(subset=["tconst"], keep="first")

print("Films :", len(data))

Films : 3508


## 3. Colonnes inutiles
Suppression des doublons IMDb/TMDB (on garde IMDb), des colonnes majoritairement vides et de celles sans utilité pour le projet.

In [17]:
data = data.drop(columns=["titleType", "isAdult", "endYear", "tagline", "id",
                          "vote_average", "vote_count", "genres_y", "spoken_languages",
                          "imdb_id", "backdrop_path", "release_date", "homepage", "adult",
                          "status", "runtime", "budget", "revenue", "title", "original_title"])

print("Colonnes :", data.shape[1])

Colonnes : 15


## 4. Titres français
Jointure avec `titres_fr.csv` (left : on garde tous les films).

In [18]:
titres_fr = pd.read_csv("../data/propre/titres_fr.csv")[["titleId", "titre_fr"]]

data = pd.merge(data, titres_fr, how="left", left_on="tconst", right_on="titleId")
data = data.drop(columns=["titleId"])

print("Films :", len(data), "| Avec titre français :", data["titre_fr"].notna().sum())

Films : 3508 | Avec titre français : 2814


## 5. Titre affiché
Titre français s'il existe ; sinon titre original (film français) ; sinon titre principal.

In [19]:
data["titre_affiche"] = np.where(data["original_language"] == "fr",
                                 data["originalTitle"],
                                 data["primaryTitle"])
data["titre_affiche"] = data["titre_fr"].fillna(data["titre_affiche"])

print("Titres vides :", data["titre_affiche"].isna().sum())

Titres vides : 0


## 6. Ordre et renommage des colonnes

In [20]:
ordre = ["tconst", "titre_affiche", "originalTitle", "primaryTitle",
         "startYear", "runtimeMinutes", "genres_x",
         "averageRating", "numVotes", "popularity",
         "original_language", "production_countries", "production_companies",
         "overview", "keywords", "poster_path"]
data = data[ordre]

data = data.rename(columns={
    "tconst": "id_imdb", "titre_affiche": "titre", "originalTitle": "titre_original",
    "primaryTitle": "titre_principal", "startYear": "annee", "runtimeMinutes": "duree_min",
    "genres_x": "genres", "averageRating": "note_imdb", "numVotes": "nb_votes_imdb",
    "popularity": "popularite_tmdb", "original_language": "langue_originale",
    "production_countries": "pays_production", "production_companies": "societes_production",
    "overview": "resume", "keywords": "mots_cles", "poster_path": "affiche",
})

print("Films :", len(data), "| Colonnes :", data.shape[1])

Films : 3508 | Colonnes : 16


## 7. Export

In [21]:
data.to_csv("../data/propre/film_complet.csv", index=False, encoding="utf-8-sig")

print("Export :", len(data), "films | Doublons :", data["id_imdb"].duplicated().sum())

Export : 3508 films | Doublons : 0


In [22]:
data.head()

,id_imdb,titre,titre_original,titre_principal,annee,duree_min,genres,note_imdb,nb_votes_imdb,popularite_tmdb,langue_originale,pays_production,societes_production,resume,mots_cles,affiche
0,tt0057485,Assassination in Rome,Il segreto del vestito rosso,Assassination in Rome,1965.0,110.0,"Comedy,Crime,Mystery",5.4,249.0,4.513,en,"France, Italy, Spain","Apo Film, Midega Film, Dicifrance","In this crime-thriller, Rome proves to be an u...",eurospy,/14NaoHCRdvScUzYbNPksNVbOEgb.jpg
1,tt0057615,Umorismo in nero,Umorismo in nero,Umorismo in nero,1965.0,93.0,Comedy,5.2,53.0,1.223,es,"Spain, Italy, France","Época Films, Sagittario Film, Lux Compagnie Ci...",NaN,dark comedy,/vMsXGeyCP6P4aGcVJRUAQAQW4kx.jpg
2,tt0057841,Memories of My Childhood,Amintiri din copilarie,Memories of My Childhood,1965.0,72.0,"Adventure,Biography,Comedy",8.0,643.0,0.618,ro,Romania,Studioul Cinematografic București,"Based on Ion Creanga's book, the story of a ch...",NaN,/cOxzgvSSms746hmX4i6SccMDfqK.jpg
3,tt0057965,Les Copains,Les copains,Les copains,1965.0,95.0,Comedy,6.2,293.0,1.288,fr,France,Les Productions de la Guéville,NaN,NaN,/kmgfML9nSXFhismrlMgEgzFJG3N.jpg
4,tt0057967,Le Corniaud,Le corniaud,The Sucker,1965.0,111.0,"Adventure,Comedy,Crime",7.3,9206.0,11.899,fr,"Spain, France, Italy","Les Films Corona, Explorer Film '58","In this Frenco-Italian gangster parody, a shop...","france, paris, france, drug smuggling, diamond...",/xkCGWJ0GYx3nK2PoFZdpl6L49q1.jpg
